<a href="https://colab.research.google.com/github/cgalla4/churn-risk-assistant/blob/main/churn_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
uploaded = files.upload()

Saving WA_Fn-UseC_-Telco-Customer-Churn.csv to WA_Fn-UseC_-Telco-Customer-Churn.csv


In [3]:
import pandas as pd
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(df.info)

<bound method DataFrame.info of       customerID  gender  SeniorCitizen Partner Dependents  tenure  \
0     7590-VHVEG  Female              0     Yes         No       1   
1     5575-GNVDE    Male              0      No         No      34   
2     3668-QPYBK    Male              0      No         No       2   
3     7795-CFOCW    Male              0      No         No      45   
4     9237-HQITU  Female              0      No         No       2   
...          ...     ...            ...     ...        ...     ...   
7038  6840-RESVB    Male              0     Yes        Yes      24   
7039  2234-XADUH  Female              0     Yes        Yes      72   
7040  4801-JZAZL  Female              0     Yes        Yes      11   
7041  8361-LTMKD    Male              1     Yes         No       4   
7042  3186-AJIEK    Male              0      No         No      66   

     PhoneService     MultipleLines InternetService OnlineSecurity  ...  \
0              No  No phone service             DSL 

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [5]:
df['Churn'].value_counts(normalize=True)

,proportion
Churn,
No,0.73463
Yes,0.26537


In [6]:
df[df["TotalCharges"].str.strip() == ""][["tenure","MonthlyCharges","TotalCharges"]]

,tenure,MonthlyCharges,TotalCharges
488,0,52.55,
753,0,20.25,
936,0,80.85,
1082,0,25.75,
1340,0,56.05,
3331,0,19.85,
3826,0,25.35,
4380,0,20.00,
5218,0,19.70,
6670,0,73.35,


In [7]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [11]:
df_model = df.drop(columns=["customerID"])
df_model["Churn"] = (df_model["Churn"] == "Yes").astype(int)
df_model = pd.get_dummies(df_model, drop_first=True)
print(df_model.shape)

(7043, 31)


In [14]:
from sklearn.model_selection import train_test_split

X = df_model.drop(columns=["Churn"])
y = df_model["Churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape, X_test.shape)

(5634, 30) (1409, 30)


In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

log_model = LogisticRegression(max_iter=5000)
log_model.fit(X_train, y_train)

log_pred = log_model.predict(X_test)
log_prob = log_model.predict_proba(X_test)[:,1]

print(classification_report(y_test, log_pred))
print("AUC:", round(roc_auc_score(y_test, log_prob),3))

              precision    recall  f1-score   support

           0       0.86      0.90      0.88      1036
           1       0.69      0.60      0.64       373

    accuracy                           0.82      1409
   macro avg       0.78      0.75      0.76      1409
weighted avg       0.82      0.82      0.82      1409

AUC: 0.862


In [16]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=300, random_state=42)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:,1]

print(classification_report(y_test, rf_pred))
print("AUC:", round(roc_auc_score(y_test, rf_prob),3))

              precision    recall  f1-score   support

           0       0.82      0.91      0.86      1036
           1       0.65      0.46      0.54       373

    accuracy                           0.79      1409
   macro avg       0.74      0.68      0.70      1409
weighted avg       0.78      0.79      0.78      1409

AUC: 0.84


In [18]:
# Churners are only ~26% of customers, so weight them more heavily to catch more of them
bal_model = LogisticRegression(max_iter=5000, class_weight="balanced")
bal_model.fit(X_train, y_train)

bal_pred = bal_model.predict(X_test)
bal_prob = bal_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, bal_pred))
print("AUC:", round(roc_auc_score(y_test, bal_prob), 3))

              precision    recall  f1-score   support

           0       0.92      0.72      0.81      1036
           1       0.52      0.83      0.64       373

    accuracy                           0.75      1409
   macro avg       0.72      0.77      0.72      1409
weighted avg       0.81      0.75      0.76      1409

AUC: 0.862


In [20]:
coefs = pd.Series(bal_model.coef_[0], index=X.columns).sort_values()

print("Biggest churn risks: ")
print(coefs.tail(8))
print("\nBiggest churn protectors: ")
print(coefs.head(8))

Biggest churn risks: 
MultipleLines_Yes                 0.288706
PhoneService_Yes                  0.295902
PaperlessBilling_Yes              0.313691
PaymentMethod_Electronic check    0.342666
StreamingTV_Yes                   0.349697
StreamingMovies_Yes               0.451669
MultipleLines_No phone service    0.521929
InternetService_Fiber optic       1.144893
dtype: float64

Biggest churn protectors: 
Contract_Two year                      -1.488019
Contract_One year                      -0.671186
OnlineSecurity_Yes                     -0.345260
TechSupport_Yes                        -0.258109
DeviceProtection_No internet service   -0.176683
OnlineBackup_No internet service       -0.176683
InternetService_No                     -0.176683
OnlineSecurity_No internet service     -0.176683
dtype: float64


In [21]:
# Show the 10 customers most likely to leave
risk_list = df.loc[X_test.index, ["customerID", "tenure", "Contract", "InternetService",
                                  "PaymentMethod", "MonthlyCharges", "TechSupport"]].copy()
risk_list["churn_prob"] = bal_prob.round(2)

top10 = risk_list.sort_values("churn_prob", ascending=False).head(10)
top10

,customerID,tenure,Contract,InternetService,PaymentMethod,MonthlyCharges,TechSupport,churn_prob
1410,7024-OHCCK,2,Month-to-month,Fiber optic,Electronic check,93.85,No,0.93
2208,7216-EWTRS,1,Month-to-month,Fiber optic,Electronic check,100.80,No,0.93
4800,9300-AGZNL,1,Month-to-month,Fiber optic,Electronic check,94.00,No,0.93
3346,2545-EBUPK,2,Month-to-month,Fiber optic,Electronic check,84.05,No,0.92
2577,4910-GMJOT,1,Month-to-month,Fiber optic,Electronic check,94.60,No,0.92
2191,3988-RQIXO,1,Month-to-month,Fiber optic,Electronic check,91.30,No,0.92
585,5192-EBGOV,1,Month-to-month,Fiber optic,Electronic check,85.70,No,0.91
2741,1724-BQUHA,5,Month-to-month,Fiber optic,Electronic check,89.50,No,0.91
3078,5277-ZLOOR,2,Month-to-month,Fiber optic,Electronic check,85.55,No,0.91
534,9282-IZGQK,1,Month-to-month,Fiber optic,Electronic check,85.00,No,0.91


In [22]:
!pip install -q -U google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.4.0 which is incompatible.


In [28]:
from google import genai
from google.colab import userdata

# Load the key from Colab Secrets so it never appears in the code
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"), http_options={"timeout": 60000})  # give up on any request after 60 seconds

response = client.models.generate_content(model="gemini-3.8-flash", contents="Say hi in 5 words")
print(response.text)

Hello there, how are you?


In [33]:
import time
from google.genai import errors

MODEL = "gemini-3.5-flash"  # change this one line if you switch models

def ask_gemini(prompt, tries=3):
    for attempt in range(tries):
        try:
            return client.models.generate_content(model=MODEL, contents=prompt).text
        except Exception as e:
            print(f"  Attempt {attempt + 1} failed: {e}")  # shows the problem instead of waiting silently
            time.sleep(10 * (attempt + 1))
    return "AI note unavailable - service busy, try again later."

def write_note(row):
    prompt = f"""You are helping a telecom account manager keep customers from leaving.
Our model says this customer has a {row.churn_prob:.0%} chance of leaving.
Details: {row.tenure} months as a customer, {row.Contract} contract, {row.InternetService} internet,
pays by {row.PaymentMethod}, ${row.MonthlyCharges}/month, tech support: {row.TechSupport}.
Our model found that fiber internet and electronic check payments raise risk,
and 1-2 year contracts and tech support lower it.
Only recommend offers from this list: 1-year contract with 10% off,
free tech support for 6 months, $5/month off for switching to autopay.
Do not invent any other prices or discounts.
Reply with exactly 3 plain-text bullets starting with "-", no intro sentence and no bold."""
    return ask_gemini(prompt)

# Send each of the 10 riskiest customers to Gemini and save the notes
notes = []
for i, (_, row) in enumerate(top10.iterrows(), start=1):
    print(f"Customer {i} of 10...")
    notes.append(write_note(row))
    time.sleep(5)  # pause between requests to stay under the free tier's limit

top10["ai_note"] = notes

# Print the first 3 to read
for _, row in top10.head(3).iterrows():
    print(row.customerID, "-", f"{row.churn_prob:.0%} chance of leaving")
    print(row.ai_note)
    print("-" * 40)

Customer 1 of 10...
Customer 2 of 10...
Customer 3 of 10...
  Attempt 1 failed: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 7h2m31.325478516s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content re

KeyboardInterrupt: 